In [ ]:
# 03_bm25_baseline.ipynb
# Цель:
# 1) показать BM25 на маленьком игрушечном примере
# 2) применить BM25 к chunks_final.csv и gold_final.csv
# 3) сохранить bm25_rankings.csv в строго заданном формате


BM25 — это классический алгоритм поиска по текстам.

Идея очень простая:


* запрос и документы разбиваются на слова;

* документ получает высокий score, если в нём встречаются слова запроса;

* редкие слова важнее частых;

* слишком длинные документы немного штрафуются.

---

* без эмбеддингов,

* без LLM,

* без генерации,

* только поиск релевантного чанка по словам вопроса.

In [5]:
import re
import math
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi # окей, будем юзать готовую реализацию BM25, а не писать свою с нуля
# Описание библиотеки rank_bm25: https://pypi.org/project/rank-bm25/. В отчете опишите как работает BM25, какие есть параметры и как они влияют на результат.
# В коде ниже показано, как использовать эту библиотеку для получения ранжирования документов по запросу.

In [6]:
def simple_tokenize(text: str) -> list[str]:
    text = str(text).lower()
    text = re.sub(r"[^a-zа-яё0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []
    return text.split()

In [7]:
sample_text = "Выручка компании за 2024 год составила 703 741 млн руб."
print(simple_tokenize(sample_text))

['выручка', 'компании', 'за', '2024', 'год', 'составила', '703', '741', 'млн', 'руб']


In [8]:
# Пример работы

toy_docs = [
    "выручка компании за 2024 год составила 703 741 млн руб",
    "чистая прибыль компании выросла на 15 процентов",
    "капитальные затраты снизились по сравнению с прошлым годом",
    "количество клиентов мобильной связи увеличилось",
]

toy_doc_ids = ["d1", "d2", "d3", "d4"]

toy_query = "какая выручка компании за 2024 год"

In [9]:
toy_tokenized_docs = [simple_tokenize(doc) for doc in toy_docs]
toy_tokenized_query = simple_tokenize(toy_query)

print("Запрос:", toy_tokenized_query)
print()

for doc_id, tokens in zip(toy_doc_ids, toy_tokenized_docs):
    print(doc_id, tokens)

Запрос: ['какая', 'выручка', 'компании', 'за', '2024', 'год']

d1 ['выручка', 'компании', 'за', '2024', 'год', 'составила', '703', '741', 'млн', 'руб']
d2 ['чистая', 'прибыль', 'компании', 'выросла', 'на', '15', 'процентов']
d3 ['капитальные', 'затраты', 'снизились', 'по', 'сравнению', 'с', 'прошлым', 'годом']
d4 ['количество', 'клиентов', 'мобильной', 'связи', 'увеличилось']


In [10]:
toy_bm25 = BM25Okapi(toy_tokenized_docs)
toy_scores = toy_bm25.get_scores(toy_tokenized_query)

toy_results = pd.DataFrame({
    "doc_id": toy_doc_ids,
    "text": toy_docs,
    "score": toy_scores,
}).sort_values("score", ascending=False).reset_index(drop=True)

toy_results

,doc_id,text,score
0,d1,выручка компании за 2024 год составила 703 741...,2.947123
1,d2,чистая прибыль компании выросла на 15 процентов,0.000000
2,d3,капитальные затраты снизились по сравнению с п...,0.000000
3,d4,количество клиентов мобильной связи увеличилось,0.000000


Мы видим, что самый высокий score получил документ про выручку.

Почему:

* в нём есть слова из запроса;

* совпадают ключевые термины: `выручка`, `компании`, `2024`, `год`;

* другие документы либо не содержат этих слов, либо содержат меньше совпадений.

Именно так BM25 и работает: он ищет документы, где слова запроса хорошо совпадают с текстом документа.

Опять же, теорию более подробно посмотрите сами и кратко опишите в отчете.

In [12]:
toy_query_2 = "чистая прибыль"
toy_scores_2 = toy_bm25.get_scores(simple_tokenize(toy_query_2))

toy_results_2 = pd.DataFrame({
    "doc_id": toy_doc_ids,
    "text": toy_docs,
    "score": toy_scores_2,
}).sort_values("score", ascending=False).reset_index(drop=True)

toy_results_2

,doc_id,text,score
0,d2,чистая прибыль компании выросла на 15 процентов,1.747006
1,d1,выручка компании за 2024 год составила 703 741...,0.000000
2,d3,капитальные затраты снизились по сравнению с п...,0.000000
3,d4,количество клиентов мобильной связи увеличилось,0.000000


In [17]:
toy_query_2 = "мобильную связь"
toy_scores_2 = toy_bm25.get_scores(simple_tokenize(toy_query_2))

toy_results_2 = pd.DataFrame({
    "doc_id": toy_doc_ids,
    "text": toy_docs,
    "score": toy_scores_2,
}).sort_values("score", ascending=False).reset_index(drop=True)

toy_results_2

,doc_id,text,score
0,d1,выручка компании за 2024 год составила 703 741...,0.0
1,d2,чистая прибыль компании выросла на 15 процентов,0.0
2,d3,капитальные затраты снизились по сравнению с п...,0.0
3,d4,количество клиентов мобильной связи увеличилось,0.0


На игрушечном примере видно:


* BM25 хорошо работает, когда слова вопроса прямо встречаются в тексте;

* дальше мы применяем ту же идею к реальным чанкам годовых отчётов.

* По умолчанию библиотека не реализует стемминг/лемматизацию для того чтобы нормально учитывать разные формы слов. Это нужно делать самостоятельно отдельным шагом препроцессинга. Идеале попробовать, можно сначала на игрушечном примере, затем на реальных данных.

In [14]:
CHUNKS_PATH = "chunks_final.csv"
GOLD_PATH = "gold_final.csv"
OUTPUT_PATH = "bm25_rankings.csv"

TOP_K = 10

In [ ]:
chunks_df = pd.read_csv(CHUNKS_PATH)
gold_df = pd.read_csv(GOLD_PATH)

print("chunks_df:", chunks_df.shape)
print("gold_df:", gold_df.shape)

chunks_df.head()
gold_df.head()

In [ ]:
required_chunk_cols = ["chunk_id", "text"]
required_gold_cols = ["query_id", "question", "relevant_chunk_id"]

missing_chunk_cols = [c for c in required_chunk_cols if c not in chunks_df.columns]
missing_gold_cols = [c for c in required_gold_cols if c not in gold_df.columns]

assert not missing_chunk_cols, f"В chunks_final.csv не хватает колонок: {missing_chunk_cols}"
assert not missing_gold_cols, f"В gold_final.csv не хватает колонок: {missing_gold_cols}"

chunks_df["chunk_id"] = chunks_df["chunk_id"].astype(str).str.strip()
chunks_df["text"] = chunks_df["text"].fillna("").astype(str).str.strip()

gold_df["query_id"] = gold_df["query_id"].astype(str).str.strip()
gold_df["question"] = gold_df["question"].fillna("").astype(str).str.strip()
gold_df["relevant_chunk_id"] = gold_df["relevant_chunk_id"].astype(str).str.strip()

chunks_df = chunks_df[chunks_df["text"] != ""].reset_index(drop=True)

print("Проверка входных файлов пройдена")

In [ ]:
def build_tokenized_corpus(chunks_df: pd.DataFrame) -> list[list[str]]:
    """
    Принимает DataFrame чанков.
    Возвращает список токенов для каждого чанка в том же порядке, что и строки chunks_df.

    Пример:
    [
        ["выручка", "компании", "за", "2024", "год"],
        ["чистая", "прибыль", "выросла"],
        ...
    ]
    """
    # TODO: реализовать
    raise NotImplementedError

In [ ]:
tokenized_corpus = build_tokenized_corpus(chunks_df)
print("Число документов в корпусе:", len(tokenized_corpus))
print("Пример токенов:", tokenized_corpus[0][:20])

In [ ]:
def build_bm25_index(tokenized_corpus: list[list[str]]) -> BM25Okapi:
    """
    Принимает tokenized_corpus и возвращает объект BM25Okapi.
    """
    # TODO: реализовать
    raise NotImplementedError

In [ ]:
bm25_index = build_bm25_index(tokenized_corpus)
print(type(bm25_index))

In [ ]:
# Для одного вопроса:

# токенизировать вопрос;
# получить BM25 score для всех чанков;
# отсортировать чанки по score по убыванию;
# вернуть top-k строк в формате таблицы.

def retrieve_top_k_bm25(
    query_id: str,
    question: str,
    bm25_index: BM25Okapi,
    chunks_df: pd.DataFrame,
    top_k: int = 10,
) -> pd.DataFrame:
    """
    Возвращает DataFrame c колонками:
    - query_id
    - method
    - rank
    - chunk_id
    - score
    """
    # TODO: реализовать
    raise NotImplementedError

In [ ]:
test_query_id = gold_df.iloc[0]["query_id"]
test_question = gold_df.iloc[0]["question"]

test_results = retrieve_top_k_bm25(
    query_id=test_query_id,
    question=test_question,
    bm25_index=bm25_index,
    chunks_df=chunks_df,
    top_k=TOP_K,
)

test_results

In [ ]:
# Для каждого вопроса из gold_df:

# вызвать retrieve_top_k_bm25(...)
# собрать все результаты в один DataFrame

def build_bm25_rankings(
    gold_df: pd.DataFrame,
    bm25_index: BM25Okapi,
    chunks_df: pd.DataFrame,
    top_k: int = 10,
) -> pd.DataFrame:
    """
    Строит итоговый ranking DataFrame для всех вопросов.
    """
    # TODO: реализовать
    raise NotImplementedError

In [ ]:
bm25_rankings_df = build_bm25_rankings(
    gold_df=gold_df,
    bm25_index=bm25_index,
    chunks_df=chunks_df,
    top_k=TOP_K,
)

print("bm25_rankings_df:", bm25_rankings_df.shape)
bm25_rankings_df.head(20)

In [ ]:
required_output_cols = ["query_id", "method", "rank", "chunk_id", "score"]
missing_output_cols = [c for c in required_output_cols if c not in bm25_rankings_df.columns]
assert not missing_output_cols, f"В bm25_rankings_df не хватает колонок: {missing_output_cols}"

assert (bm25_rankings_df["method"] == "bm25").all(), "Колонка method должна быть равна 'bm25'"
assert (bm25_rankings_df["rank"] >= 1).all(), "rank должен начинаться с 1"
assert bm25_rankings_df["chunk_id"].notna().all(), "Есть пустые chunk_id"
assert bm25_rankings_df["query_id"].notna().all(), "Есть пустые query_id"

print("Проверка bm25_rankings_df пройдена")

In [ ]:
# Визуально посмотрим
sample_query_ids = gold_df["query_id"].head(3).tolist()

for qid in sample_query_ids:
    print("=" * 80)
    print("QUERY_ID:", qid)
    print("QUESTION:", gold_df.loc[gold_df["query_id"] == qid, "question"].iloc[0])
    bm25_rankings_df[bm25_rankings_df["query_id"] == qid].head(10)

In [ ]:
bm25_rankings_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"Сохранён файл: {OUTPUT_PATH}")